# Stage 1 — SAE Setup & Verification
**Owner: Person A** | Week 1

## Goal
Load DINOv2 ViT-B/14-reg via Prisma, load the SAE for layer 11,
and verify quality targets before anything else is built on top of it.

## Rule
No logic in cells. Every substantive function lives in .
A cell should call one function and check its output.

## Outputs
- Quality report: L0, reconstruction loss, dead feature fraction
- Confirmation that all targets in  are met
- Cache build handed off to Person C

In [ ]:
# Install (run once on Colab)
# !pip install -r ../requirements.txt
import sys; sys.path.append("..")

## 1. Load config

In [ ]:
from src.config import get_config
cfg = get_config()
print("Model:", cfg.model.name)
print("SAE primary layer:", cfg.sae.primary_layer)

## 2. Load model and run smoke test

In [ ]:
from src.model import get_model, get_device, preprocess_image

device = get_device()
model = get_model()

# TODO: set a real image path after downloading ImageNet val
TEST_IMAGE = "../data/imagenet_val/n02007558/REPLACE_WITH_REAL_IMAGE.JPEG"
pixel_values = preprocess_image(TEST_IMAGE).to(device)
logits, act_cache = model.run_with_cache(pixel_values)

resid = act_cache[f"blocks.{cfg.sae.primary_layer}.hook_resid_post"]
print("Logits shape:", logits.shape)          # expect (1, 1000)
print("Residual stream shape:", resid.shape)  # expect (1, 201, 768)

## 3. Load SAE and verify quality targets

In [ ]:
from src.sae import get_sae, get_l0_sparsity, get_reconstruction_loss

sae = get_sae(layer=cfg.sae.primary_layer)

l0   = get_l0_sparsity(resid, layer=cfg.sae.primary_layer)
recon = get_reconstruction_loss(resid, layer=cfg.sae.primary_layer)

print(f"L0 sparsity:         {l0:.1f}  (target < {cfg.sae.l0_target})")
print(f"Reconstruction loss:  {recon:.4f}  (target < 0.05)")

assert l0 < cfg.sae.l0_target,  f"FAIL: L0 too high ({l0:.1f})"
assert recon < 0.05,             f"FAIL: recon loss too high ({recon:.4f})"
print("All quality checks passed.")

## 4. Verify ablate_feature end-to-end

In [ ]:
from src.sae import encode, ablate_feature

features = encode(resid, layer=cfg.sae.primary_layer)
print("SAE feature shape:", features.shape)  # (1, 201, d_sae)

top_feature = int(features[0].max(dim=0).values.argmax())
ablated = ablate_feature(resid, feature_idx=top_feature,
                         layer=cfg.sae.primary_layer)
assert ablated.shape == resid.shape
print(f"Ablated feature #{top_feature}. Output shape matches input: OK")

## 5. Hand off to Person C: build activation cache

Once the above cells pass, tell Person C to run .
The cache build requires the full ImageNet val path set in .